## Import Libraries

In [1]:
import numpy as np
import pandas as pd
import re

## Import Dataset

In [2]:
sheets = pd.ExcelFile('Swap and Swaption Markets_Amended.xlsx').sheet_names
sheets

['LIBOR (legacy)', 'OIS (SOFR)', 'OIS (Term SOFR)', 'Swaption']

In [3]:
df_LIBOR = pd.read_excel('Swap and Swaption Markets_Amended.xlsx', sheet_name=sheets[0])
df_LIBOR;

Notes on LIBOR Rates on first inspection:
* Day Count Convention: 30/360
* Floating Rate: LIBOR 6M
* Floating Leg Frequency and Fixed Leg Frequency: Semi-Annual

In [4]:
df_LIBOR = df_LIBOR.dropna(axis=1)

In [5]:
def convert_tenor_to_numeric(series):
    num = series.str.slice(stop = -1).astype(int)
    freq = series.str.slice(start = -1)

    conditions = [freq.str.upper() == 'M', freq.str.upper() == 'Y']
    choices = [num * 30 / 360, num.astype(float)]

    return np.select(conditions, choices, default=np.nan)

In [6]:
df_LIBOR['Tenor_Year_Fraction'] = convert_tenor_to_numeric(df_LIBOR['Tenor'])
df_LIBOR

,Tenor,Product,Rate,Tenor_Year_Fraction
0,6m,LIBOR,0.0450,0.5
1,1y,IRS,0.0480,1.0
2,2y,IRS,0.0500,2.0
3,3y,IRS,0.0515,3.0
4,4y,IRS,0.0525,4.0
5,5y,IRS,0.0530,5.0
6,7y,IRS,0.0550,7.0
7,10y,IRS,0.0570,10.0
8,15y,IRS,0.0600,15.0
9,20y,IRS,0.0650,20.0


In [7]:
df_OIS = pd.read_excel('Swap and Swaption Markets_Amended.xlsx', sheet_name=sheets[1])
df_OIS;

Notes on OIS Rates on first inspection:
* Day Count Convention: 30/360
* Floating Rate: SOFR 1D
* Overnight Leg Frequency and Fixed Leg Frequency: Annual

In [8]:
df_OIS = df_OIS.dropna(axis = 1)
# df_OIS['Term'].str.split()[1]

In [9]:
def convert_tenor_to_numeric_OIS(series):
    parts = series.str.split()
    num = parts.str[0].astype(int)
    unit = parts.str[1]

    conditions = [unit == 'WK', unit == 'MO', unit == 'YR']
    choices = [num * 30 / (4 * 360), num * 30 / 360, num.astype(float)]

    return np.select(conditions, choices, default=np.nan)

In [10]:
df_OIS['Tenor_Year_Numeric'] = convert_tenor_to_numeric_OIS(df_OIS['Term'])
df_OIS

,Term,Market Rate,Tenor_Year_Numeric
0,1 WK,3.65950,0.020833
1,2 WK,3.65801,0.041667
2,3 WK,3.66100,0.062500
3,1 MO,3.67010,0.083333
4,2 MO,3.66600,0.166667
5,3 MO,3.65950,0.250000
6,4 MO,3.64900,0.333333
7,5 MO,3.63505,0.416667
8,6 MO,3.61610,0.500000
9,7 MO,3.59455,0.583333


In [11]:
df_OIS_Term = pd.read_excel('Swap and Swaption Markets_Amended.xlsx', sheet_name=sheets[2])
# df_OIS_Term;

Notes on OIS Term Rates on first inspection:
* Day Count Convention: 30/360
* Floating Rate: SOFR 3M
* Overnight Leg Frequency and Fixed Leg Frequency: Annual

In [12]:
df_OIS_Term = df_OIS_Term.dropna(axis = 1)
df_OIS_Term['Tenor_Year_Numeric'] = convert_tenor_to_numeric(df_OIS_Term['Tenor'])
df_OIS_Term

,Tenor,Rate,Tenor_Year_Numeric
0,1Y,0.0465,1.0
1,2Y,0.0435,2.0
2,3Y,0.0415,3.0
3,4Y,0.0402,4.0
4,5Y,0.0395,5.0
5,6Y,0.0392,6.0
6,7Y,0.0390,7.0
7,8Y,0.0390,8.0
8,9Y,0.0391,9.0
9,10Y,0.0393,10.0


In [13]:
df_swaption = pd.read_excel('Swap and Swaption Markets_Amended.xlsx', sheet_name=sheets[3])
df_swaption;

Notes on Swaption Data on first inspection:
* This is lognormal Implied Vol for SOFR Swaptions
* Convection for bps strike is (Forward + Basis Point)

In [14]:
df_swaption = pd.read_excel('Swap and Swaption Markets_Amended.xlsx', sheet_name=sheets[3], header = 2)
df_swaption['Expiry_Year'] = convert_tenor_to_numeric(df_swaption['Expiry'])
df_swaption['Tenor_Year_Fraction'] = convert_tenor_to_numeric(df_swaption['Tenor'])
df_swaption

,Expiry,Tenor,-200bps,-150bps,-100bps,-50bps,-25bps,ATM,+25bps,+50bps,+100bps,+150bps,+200bps,Expiry_Year,Tenor_Year_Fraction
0,1Y,1Y,91.570,62.030,44.130,31.224,26.182,22.50,20.96,21.40,24.34,27.488,30.297,1.0,1.0
1,1Y,2Y,83.270,61.240,46.570,35.807,31.712,28.72,27.12,26.84,28.51,31.025,33.523,1.0,2.0
2,1Y,3Y,73.920,56.870,44.770,35.745,32.317,29.78,28.29,27.80,28.77,30.725,32.833,1.0,3.0
3,1Y,5Y,55.190,44.640,36.510,30.242,27.851,26.07,24.98,24.56,25.12,26.536,28.165,1.0,5.0
4,1Y,10Y,41.180,35.040,30.207,26.619,25.351,24.47,23.98,23.82,24.25,25.204,26.355,1.0,10.0
5,5Y,1Y,67.800,49.090,38.400,31.485,29.060,27.26,26.04,25.32,24.94,25.320,25.980,5.0,1.0
6,5Y,2Y,57.880,46.410,39.033,33.653,31.531,29.83,28.56,27.65,26.71,26.540,26.760,5.0,2.0
7,5Y,3Y,53.430,44.440,38.180,33.437,31.536,29.98,28.76,27.82,26.67,26.200,26.150,5.0,3.0
8,5Y,5Y,41.990,36.524,32.326,29.005,27.677,26.60,25.73,25.02,24.06,23.570,23.400,5.0,5.0
9,5Y,10Y,34.417,30.948,28.148,25.954,25.136,24.51,23.99,23.56,22.91,22.490,22.250,5.0,10.0


## Part 1: Bootstrapping Swap Curves

### A: LIBOR Bootstrapped Discount Factor